# Proyecto Final: Análisis de Telemetría, Estrategia de Carrera y Rendimiento Histórico en la Fórmula 1 

## 1. Introducción y carga de datos
En este notebook realizaremos el proceso de carga, estudio de Calidad, Limpieza y Transformación (ETL) de los datos del campeonato mundial de Fórmula 1.

Cargamos las librerías principales (`pandas`, `numpy`) y leemos los 7 archivos `.csv` en bruto ubicados en la carpeta `rawdata/`. 

> **Nota técnica en la lectura de los datos:** Ya que estos han sido cargados desde un repositorio público se ha observado la presencia de diferentes valores nulos o no rellenos, la nomenclarura de estos hay ocasiones que variaba, por lo que se ha decidido añadir: `na_values='\\N'` para que Pandas reconozca automáticamente las secuencias `\N` provenientes del fichero original como valores nulos (`NaN`).

### 1.1. Configuración del Entorno de Desarrollo (Virtual Environment)

Para garantizar la estabilidad, reproductibilidad y un aislamiento adecuado del proyecto, se ha creado y configurado un **entorno virtual de Python (`.venv`)**. Debido a que he tenido distintos problemas al pasar de un dispositivo a otro, ya que el proyecto ha sido realziado en dos ordenadores simultáneamente, he decidido hacer un entorno para garantizar: 

1. **Aislamiento de Dependencias y Evitación de Conflictos:** 
   El sistema base o las instalaciones globales de Python a menudo presentan incompatibilidades entre versiones de librerías previamente instaladas.

2. **Reproductibilidad Técnica:** 
   Permite que cualquier otro analista o docente pueda ejecutar el proyecto en un entorno idéntico utilizando el archivo, y de esta manera asegurar que el código funcione de la misma manera en cualquier sistema operativo.

### 1.2. Librerías necesarias

En esta sección se importan las librerías fundamentales de Python necesarias para todo el flujo de trabajo (las vistas en clase)
- **Pandas** 
- **NumPy**


Ambas son necesarias para  la manipulación, limpieza y agregación de estructuras de datos. Además cargadas usando los alias comúnmente usados, `pd` y `np` respectivamente.

In [35]:
#!pip3 install numpy 
import numpy as np
#!pip3 install pandas
import pandas as pd

In [36]:
# Cargar los 7 archivos interpretando 
results = pd.read_csv('rawdata/results.csv', na_values='\\N')
races = pd.read_csv('rawdata/races.csv', na_values='\\N')
drivers = pd.read_csv('rawdata/drivers.csv', na_values='\\N')
constructors = pd.read_csv('rawdata/constructors.csv', na_values='\\N')
circuits = pd.read_csv('rawdata/circuits.csv', na_values='\\N')
lap_times = pd.read_csv('rawdata/lap_times.csv', na_values='\\N')
pit_stops = pd.read_csv('rawdata/pit_stops.csv', na_values='\\N')

In [37]:
# Ver una primera vista de los datos
results.head()
races.head()
drivers.head()
constructors.head()
circuits.head()
lap_times.head()
pit_stops.head()

,raceId,driverId,stop,lap,time,duration,milliseconds
0,258,100,1,1,14:01:34,49.111,49111.0
1,258,79,1,17,14:20:46,28.482,28482.0
2,258,57,1,18,14:22:35,43.745,43745.0
3,258,71,1,18,14:23:00,21.992,21992.0
4,258,105,1,19,14:24:39,27.693,27693.0


## 2. Creación del dataset final

### 2.1 Justificación 
En lugar de analizar la calidad e integridad de los 7 archivos de forma aislada, se ha optado por aplicar una **estrategia de integración previa**. Los puntos qeu han hecho que parezca esta la opción óptima son: 

1. **Visión y contexto completo:** Analizar variables como tiempos de vuelta o duraciones de *pit stops* de forma independiente carece de contexto sin vincular los datos del piloto, el monoplaza, la temporada y la geografía del circuito.
2. **Eficiencia en el Flujo de Limpieza:** Permite centralizar la detección de duplicados, la inspección de valores nulos y la conversión de tipos sobre **un único dataframe**, optimizando el código ejecutado en Python.

3. **Validación Inmediata de Requisitos:** Garantiza desde la primera fase que el volumen resultante cumple las condiciones exigidas por el proyecto (**>50.000 filas y >20 columnas**).



### 2.2 Arquitectura de Uniones (Merge Relacional)

La unificación de las 7 fuentes se ha estructurado en tres niveles:

* **Paso 1 :** Se integran la tabla `results` con `races`, `drivers`, `constructors` y `circuits` mediante sus claves primarias (`raceId`, `driverId`, `constructorId`, `circuitId`).
* **Paso 2 :** Se enlaza la tabla resultante del paso 1 con `lap_times` (+870.000 filas) utilizando la clave compuesta **`[raceId, driverId]`**.
* **Paso 3:** Se realiza un **`LEFT JOIN`** con `pit_stops` sobre **`[raceId, driverId, lap]`**. El uso del *Left Join* es necesario para preservar las vueltas estándar en las que no hubo parada,(se ha probado) asignando valores nulos únicamente a la maniobra de boxes.


In [38]:
# 2. Merge de datos pilotos/escuderia/circuito/carrera y Resultados
data = results.merge(races[['raceId', 'year', 'round', 'circuitId', 'name', 'date']], on='raceId')
data = data.merge(drivers[['driverId', 'driverRef', 'code', 'forename', 'surname', 'dob', 'nationality']], on='driverId')
data = data.merge(constructors[['constructorId', 'constructorRef', 'name', 'nationality']], on='constructorId', suffixes=('_driver', '_constructor'))
data = data.merge(circuits[['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt']], on='circuitId', suffixes=('_race', '_circuit'))

# Renombrar columnas para evitar ambigüedades
data = data.rename(columns={
    'name_race': 'race_name',
    'name_constructor': 'constructor_name',
    'name': 'circuit_name',
    'nationality_driver': 'driver_nationality',
    'nationality_constructor': 'constructor_nationality'
})

# 3. Merge con Telemetría por Vuelta (lap_times) y Paradas en Boxes (pit_stops)
data_final = lap_times.merge(data, on=['raceId', 'driverId'], suffixes=('_lap', '_result'))
data_final = data_final.merge(
    pit_stops[['raceId', 'driverId', 'lap', 'stop', 'duration', 'milliseconds']].rename(
        columns={'duration': 'pit_duration', 'milliseconds': 'pit_milliseconds'}), 
    on=['raceId', 'driverId', 'lap'], how='left')


data_final.head()


# Exportar el fichero combinado
ruta_salida = 'data_combinada.csv'
data_final.to_csv(ruta_salida, index=False)


In [39]:
print(f"Dimensiones finales: {data_final.shape[0]:,} filas y {data_final.shape[1]} columnas.")

Dimensiones finales: 879,870 filas y 46 columnas.


## 3. Calidad de los datos y Conteo de Nulos

Al ser un data set resultante de dimensiones tan grandes (879.870 filas y 46 columnas) se va a proceder a un análisis por bloque para garantizar que no se deja nada atrás.

### 3.1 Limpieza — Bloque 1: Pilotos y Constructores

En este primer bloque analizamos de forma independiente las variables demográficas e identificadoras de los pilotos y las escuderías:
* **Pilotos:** `forename`, `surname`, `dob` (fecha de nacimiento), `driver_nationality`, `code`, `number`.
* **Constructores:** `constructor_name`, `constructor_nationality`.

In [40]:
#Inspección de nulos específica del Bloque 1 -- > coger las columnas específicas
cols_bloque1 = ['forename', 'surname', 'dob', 'driver_nationality', 'code', 'number', 'constructor_name', 'constructor_nationality']
print(data_final[cols_bloque1].isnull().sum())


forename                        0
surname                         0
dob                             0
driver_nationality              0
code                       309369
number                          0
constructor_name                0
constructor_nationality         0
dtype: int64


#### Hallazgos y Decisiones:
1. **Dorsales y Códigos (`code`):** Presenta un alto porcentaje de valores nulos, tras investigar un poco en el asunto puede concluirse que esto es debido a que el sistema de dorsales fijos y abreviaturas de 3 letras se instauró oficialmente en la F1 en **2014**.
2. **Generación de Identificador Único (`driver_full_name`):** Para evitar inconsistencias al trabajar con pilotos históricos previos a 2014, creamos la columna concatenada del nombre completo para identificar de manera completa a cada piloto.
3. **Conversión de Fechas y Cálculo de Edad (`driver_age`):** Convertimos `dob` (date of birth) a formato fecha para calcular la edad exacta del piloto al momento de disputar cada Gran Premio.

In [41]:

# Crear nombre completo del piloto 
data_final['driver_full_name'] = data_final['forename'] + ' ' + data_final['surname']

# 3. Conversión de fecha de nacimiento y cálculo de edad del piloto en la carrera
data_final['dob'] = pd.to_datetime(data_final['dob'])
data_final['date'] = pd.to_datetime(data_final['date'])
data_final['driver_age'] = (data_final['date'] - data_final['dob']).dt.days // 365

#Vemos un ejemplo de que ha funcionado bien los cambios que hemos hecho
data_final[['driver_full_name', 'driver_nationality', 'driver_age', 'constructor_name']].head(3)

,driver_full_name,driver_nationality,driver_age,constructor_name
0,Nelson Piquet,Brazilian,30,Brabham
1,Nelson Piquet,Brazilian,30,Brabham
2,Nelson Piquet,Brazilian,30,Brabham


### 3.2  Limpieza — Bloque 2: Telemetría y Rendimiento en Pista

En este bloque analizamos las variables de alta frecuencia relacionadas con el ritmo de carrera, posiciones y resultados:
* **Métricas por vuelta:** `lap`, `position_lap`, `time_lap`, `milliseconds_lap`.
* **Métricas de clasificación y resultado final:** `grid` (posición de salida), `positionOrder` (posición final oficial), `points`, `statusId` (causa de finalización/abandono).

In [42]:
# Inspección de nulos en Bloque 2
cols_bloque2 = ['lap', 'position_lap', 'milliseconds_lap', 'grid', 'positionOrder', 'points', 'statusId']
print(data_final[cols_bloque2].isnull().sum())

lap                    0
position_lap           0
milliseconds_lap       0
grid                1066
positionOrder          0
points                 0
statusId               0
dtype: int64


#### Hallazgos y Decisiones:
1. **Parrilla de salida (`grid`):** Presenta ~1.000 nulos asociados a salidas desde el callejón de boxes (*pit lane*) o sanciones administrativas. Imputamos con el valor máximo de la parrilla de esa carrera.
2. **Conversión a Segundos (`lap_seconds`):** Convertimos `milliseconds_lap` a segundos dividiendo entre 1.000 para facilitar interpretaciones y medias de tiempo.
3. **Métrica de Remontada (`positions_gained`):** Calculamos la diferencia neta entre la posición de salida y la posición final (`grid - positionOrder`).

In [43]:
# Cambiamos nulos en 'grid' (salidas desde pit lane) con la peor posición de salida de la carrera o si no 20
data_final['grid'] = data_final.groupby('raceId')['grid'].transform(lambda x: x.fillna(x.max() if pd.notnull(x.max()) else 20))

# Conversión de tiempos de vuelta a segundos
data_final['lap_seconds'] = data_final['milliseconds_lap'] / 1000.0

# Cálculo de posiciones ganadas/perdidas 
data_final['positions_gained'] = data_final['grid'] - data_final['positionOrder']

#Ejemplo
data_final[['lap', 'position_lap', 'lap_seconds', 'grid', 'positionOrder', 'positions_gained']].head(3)

,lap,position_lap,lap_seconds,grid,positionOrder,positions_gained
0,1,1,102.085,1.0,11,-10.0
1,2,2,96.287,1.0,11,-10.0
2,3,2,94.627,1.0,11,-10.0


### 3.3 Auditoría — Bloque 3: Estrategia y Paradas en Boxes

Analizamos el rendimiento de los monoplaazas en la calle de boxes:
* **Variables:** `stop` (número de parada), `pit_duration` (duración en texto), `pit_milliseconds` (duración en ms).

In [44]:
null_bloque3 = (data_final['stop'].isnull().sum())
porcentaje_null = round((null_bloque3/len(data_final))*100,4)
print(f'Hay un total de {null_bloque3} nulos de vueltas con paradas en pit, un {porcentaje_null} %')

Hay un total de 857267 nulos de vueltas con paradas en pit, un 97.4311 %



#### Hallazgos y Decisiones:
1. **Presencia masiva de nulos (~97,4%):** Como se realizó un *Left Join*, el 97,4% de los registros corresponden a vueltas estándar sin parada en boxes.
2. **Indicador de Parada (`is_pit_stop`):** Generamos un flag booleano (`True`/`False`) para aislar instantáneamente las vueltas en las que el vehículo realizó un *pit stop*.
3. **Conversión de Duración de Parada (`pit_seconds`):** Creamos una variable numérica continua dividiendo `pit_milliseconds` entre 1.000.

In [45]:
# 1. Crear indicador booleano de si hubo pit stop en esa vuelta
data_final['is_pit_stop'] = data_final['stop'].notnull()

# 2. Convertir duración del pit stop a segundos (solo para vueltas con parada)
data_final['pit_seconds'] = data_final['pit_milliseconds'] / 1000.0

print(f"Vueltas totales analizadas: {len(data_final)}")
print(f"Vueltas con parada en boxes: {data_final['is_pit_stop'].sum()}")

Vueltas totales analizadas: 879870
Vueltas con parada en boxes: 22603



## 4. Comprobacion final y exportar el documento:

Vemos exactamente como es el dataset final con el que se va a realizar el análisis y exportamos el mismo en _.csv_ para guardarlo.


In [47]:
# Exportar el fichero final 
ruta_salida = 'f1_data.csv'
data_final.to_csv(ruta_salida, index=False)


KeyboardInterrupt: 

In [50]:
data_final.info()
print(f"Dimensiones finales: {data_final.shape[0]:,} filas y {data_final.shape[1]} columnas.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 879870 entries, 0 to 879869
Data columns (total 52 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   raceId                   879870 non-null  int64         
 1   driverId                 879870 non-null  int64         
 2   lap                      879870 non-null  int64         
 3   position_lap             879870 non-null  int64         
 4   time_lap                 879870 non-null  object        
 5   milliseconds_lap         879870 non-null  int64         
 6   resultId                 879870 non-null  int64         
 7   constructorId            879870 non-null  int64         
 8   number                   879870 non-null  float64       
 9   grid                     879870 non-null  float64       
 10  position_result          741318 non-null  float64       
 11  positionText             879870 non-null  object        
 12  positionOrder   